# Email Campaign Experimentation Analysis

This notebook evaluates a randomized email marketing experiment using treatment-control comparisons, conversion lift, revenue lift, bootstrap confidence intervals, and segment-level treatment effect analysis.

The business goal is to determine whether promotional email campaigns increased visits, conversions, and revenue compared with a no-email control group.

## 1. Load Dataset

The dataset contains customer-level pre-campaign attributes, randomized campaign assignments, and post-campaign outcomes.

Key outcome variables:

- `visit`: whether the customer visited after the campaign
- `conversion`: whether the customer purchased
- `spend`: customer spend after the campaign

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

DATA_PATH = Path("../data/raw/mine_that_data_email.csv")

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(64000, 12)


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


In [2]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

df.head()

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


## 2. Data Quality Checks

This section validates the dataset shape, column types, missing values, treatment group sizes, and outcome distributions before estimating treatment effects.

In [3]:
df.info()

print(df["segment"].value_counts())
print(df["conversion"].value_counts())
print(df["visit"].value_counts())

missing = (
    df.isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

missing.columns = ["column", "missing_rate"]
missing

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recency          64000 non-null  int64  
 1   history_segment  64000 non-null  object 
 2   history          64000 non-null  float64
 3   mens             64000 non-null  int64  
 4   womens           64000 non-null  int64  
 5   zip_code         64000 non-null  object 
 6   newbie           64000 non-null  int64  
 7   channel          64000 non-null  object 
 8   segment          64000 non-null  object 
 9   visit            64000 non-null  int64  
 10  conversion       64000 non-null  int64  
 11  spend            64000 non-null  float64
dtypes: float64(2), int64(6), object(4)
memory usage: 5.9+ MB
segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: int64
conversion
0    63422
1      578
Name: count, dtype: int64
visit
0    54606
1     

,column,missing_rate
0,recency,0.0
1,history_segment,0.0
2,history,0.0
3,mens,0.0
4,womens,0.0
5,zip_code,0.0
6,newbie,0.0
7,channel,0.0
8,segment,0.0
9,visit,0.0


## 3. Experiment Design Review

The no-email group is treated as the control group. The Mens E-Mail and Womens E-Mail groups are treated as randomized campaign variants.

This section summarizes users, visits, conversions, revenue, and revenue per user by experiment group.

In [4]:
experiment_design = (
    df.groupby("segment")
    .agg(
        users=("conversion", "size"),
        conversion_rate=("conversion", "mean"),
        visit_rate=("visit", "mean"),
        avg_spend_per_user=("spend", "mean"),
        total_revenue=("spend", "sum"),
        avg_spend_among_buyers=("spend", lambda x: x[x > 0].mean())
    )
    .reset_index()
)

experiment_design

,segment,users,conversion_rate,visit_rate,avg_spend_per_user,total_revenue,avg_spend_among_buyers
0,Mens E-Mail,21307,0.012531,0.182757,1.422617,30311.69,113.526929
1,No E-Mail,21306,0.005726,0.106167,0.652789,13908.33,114.002705
2,Womens E-Mail,21387,0.008837,0.151400,1.077202,23038.11,121.894762


## 4. Treatment Group Setup

This section creates treatment indicators for comparing each email campaign against the no-email control group.

The main comparisons are:

- Mens E-Mail vs No E-Mail
- Womens E-Mail vs No E-Mail
- Any E-Mail vs No E-Mail

In [5]:
df["any_email"] = np.where(df["segment"] == "No E-Mail", 0, 1)

df["mens_email"] = np.where(df["segment"] == "Mens E-Mail", 1, 0)
df["womens_email"] = np.where(df["segment"] == "Womens E-Mail", 1, 0)

df["is_control"] = np.where(df["segment"] == "No E-Mail", 1, 0)

df["treatment_group"] = np.where(
    df["segment"] == "No E-Mail",
    "Control",
    "Treatment"
)

In [6]:
group_summary = (
    df.groupby("segment")
    .agg(
        users=("conversion", "size"),
        visitors=("visit", "sum"),
        buyers=("conversion", "sum"),
        total_revenue=("spend", "sum"),
        conversion_rate=("conversion", "mean"),
        visit_rate=("visit", "mean"),
        revenue_per_user=("spend", "mean"),
        revenue_per_buyer=("spend", lambda x: x[x > 0].mean())
    )
    .reset_index()
)

group_summary["conversion_rate"] = group_summary["conversion_rate"] * 100
group_summary["visit_rate"] = group_summary["visit_rate"] * 100

group_summary

,segment,users,visitors,buyers,total_revenue,conversion_rate,visit_rate,revenue_per_user,revenue_per_buyer
0,Mens E-Mail,21307,3894,267,30311.69,1.253109,18.275684,1.422617,113.526929
1,No E-Mail,21306,2262,122,13908.33,0.572609,10.616728,0.652789,114.002705
2,Womens E-Mail,21387,3238,189,23038.11,0.883714,15.140038,1.077202,121.894762


## 5. Treatment vs Control Lift Analysis

This section estimates campaign impact using conversion lift, revenue-per-user lift, incremental buyers, incremental revenue, and two-proportion z-tests for conversion-rate differences.

In [7]:
from statsmodels.stats.proportion import proportions_ztest
import numpy as np
import pandas as pd

def compare_groups(data, treatment_name, control_name="No E-Mail"):
    treatment = data[data["segment"] == treatment_name]
    control = data[data["segment"] == control_name]
    
    n_t = len(treatment)
    n_c = len(control)
    
    conv_t = treatment["conversion"].sum()
    conv_c = control["conversion"].sum()
    
    rate_t = conv_t / n_t
    rate_c = conv_c / n_c
    
    count = np.array([conv_t, conv_c])
    nobs = np.array([n_t, n_c])
    
    z_stat, p_value = proportions_ztest(count, nobs)
    
    rpu_t = treatment["spend"].mean()
    rpu_c = control["spend"].mean()
    
    absolute_conversion_lift = rate_t - rate_c
    relative_conversion_lift = (rate_t / rate_c) - 1
    
    incremental_buyers = absolute_conversion_lift * n_t
    incremental_revenue = (rpu_t - rpu_c) * n_t
    
    return {
        "comparison": f"{treatment_name} vs {control_name}",
        "treatment_users": n_t,
        "control_users": n_c,
        "treatment_buyers": int(conv_t),
        "control_buyers": int(conv_c),
        "treatment_conversion_rate": rate_t,
        "control_conversion_rate": rate_c,
        "absolute_conversion_lift": absolute_conversion_lift,
        "relative_conversion_lift": relative_conversion_lift,
        "conversion_z_stat": z_stat,
        "conversion_p_value": p_value,
        "treatment_revenue_per_user": rpu_t,
        "control_revenue_per_user": rpu_c,
        "revenue_per_user_lift": rpu_t - rpu_c,
        "incremental_buyers": incremental_buyers,
        "incremental_revenue": incremental_revenue
    }

### Overall Campaign Lift Results

The table below compares each treatment group against the no-email control group.

Conversion-rate columns are shown as percentages for readability.

In [8]:
comparisons = pd.DataFrame([
    compare_groups(df, "Mens E-Mail"),
    compare_groups(df, "Womens E-Mail")
])

any_email_df = df.copy()
any_email_df["segment"] = np.where(
    any_email_df["segment"] == "No E-Mail",
    "No E-Mail",
    "Any E-Mail"
)

any_email_result = compare_groups(any_email_df, "Any E-Mail")

comparisons = pd.concat(
    [comparisons, pd.DataFrame([any_email_result])],
    ignore_index=True
)

comparison_display = comparisons.copy()

rate_cols = [
    "treatment_conversion_rate",
    "control_conversion_rate",
    "absolute_conversion_lift",
    "relative_conversion_lift"
]

for col in rate_cols:
    comparison_display[col] = comparison_display[col] * 100

comparison_display

,comparison,treatment_users,control_users,treatment_buyers,control_buyers,treatment_conversion_rate,control_conversion_rate,absolute_conversion_lift,relative_conversion_lift,conversion_z_stat,conversion_p_value,treatment_revenue_per_user,control_revenue_per_user,revenue_per_user_lift,incremental_buyers,incremental_revenue
0,Mens E-Mail vs No E-Mail,21307,21306,267,122,1.253109,0.572609,0.680501,118.842188,7.385114,1.523224e-13,1.422617,0.652789,0.769827,144.994274,16402.707211
1,Womens E-Mail vs No E-Mail,21387,21306,189,122,0.883714,0.572609,0.311106,54.331304,3.779561,1.571051e-04,1.077202,0.652789,0.424412,66.536187,9076.904062
2,Any E-Mail vs No E-Mail,42694,21306,456,122,1.068066,0.572609,0.495457,86.526306,6.243765,4.271614e-10,1.249585,0.652789,0.596796,211.530461,25479.611273


## 6. Revenue Lift Confidence Intervals

Customer spend is highly skewed because most customers spend zero. To estimate uncertainty around revenue-per-user lift, this section uses bootstrap confidence intervals instead of relying only on parametric tests.

In [9]:
def bootstrap_mean_diff(treatment_values, control_values, n_bootstrap=5000, random_state=42):
    rng = np.random.default_rng(random_state)

    treatment_values = np.array(treatment_values)
    control_values = np.array(control_values)

    boot_diffs = []

    for _ in range(n_bootstrap):
        treatment_sample = rng.choice(
            treatment_values,
            size=len(treatment_values),
            replace=True
        )

        control_sample = rng.choice(
            control_values,
            size=len(control_values),
            replace=True
        )

        boot_diffs.append(treatment_sample.mean() - control_sample.mean())

    lower, upper = np.percentile(boot_diffs, [2.5, 97.5])

    return {
        "mean_diff": np.mean(boot_diffs),
        "ci_lower": lower,
        "ci_upper": upper
    }

### Bootstrap Revenue Lift Results

The confidence intervals below estimate the plausible range of revenue-per-user lift for each campaign comparison.

A confidence interval fully above zero suggests a positive revenue impact.

In [10]:
bootstrap_results = []

for treatment_name in ["Mens E-Mail", "Womens E-Mail"]:
    treatment = df[df["segment"] == treatment_name]["spend"]
    control = df[df["segment"] == "No E-Mail"]["spend"]
    
    result = bootstrap_mean_diff(treatment, control)
    result["comparison"] = f"{treatment_name} vs No E-Mail"
    bootstrap_results.append(result)

treatment = df[df["segment"] != "No E-Mail"]["spend"]
control = df[df["segment"] == "No E-Mail"]["spend"]

result = bootstrap_mean_diff(treatment, control)
result["comparison"] = "Any E-Mail vs No E-Mail"
bootstrap_results.append(result)

bootstrap_revenue_lift = pd.DataFrame(bootstrap_results)

bootstrap_revenue_lift = bootstrap_revenue_lift[
    ["comparison", "mean_diff", "ci_lower", "ci_upper"]
]

bootstrap_revenue_lift

,comparison,mean_diff,ci_lower,ci_upper
0,Mens E-Mail vs No E-Mail,0.770057,0.479081,1.049206
1,Womens E-Mail vs No E-Mail,0.423360,0.158774,0.684408
2,Any E-Mail vs No E-Mail,0.597576,0.378516,0.817245


## 7. Segment-Level Treatment Effects

This section evaluates whether treatment effects vary across customer segments.

Segments analyzed:

- Historical spend segment
- Zip-code type
- Customer channel
- New-customer flag

Segment-level p-values are used to distinguish statistically reliable effects from directional patterns.

In [11]:
def segment_lift_with_significance(
    data,
    segment_col,
    treatment_name,
    control_name="No E-Mail",
    min_users=200
):
    rows = []

    for segment_value, segment_df in data.groupby(segment_col):
        treatment = segment_df[segment_df["segment"] == treatment_name]
        control = segment_df[segment_df["segment"] == control_name]

        if len(treatment) < min_users or len(control) < min_users:
            continue

        conv_t_count = treatment["conversion"].sum()
        conv_c_count = control["conversion"].sum()

        n_t = len(treatment)
        n_c = len(control)

        conv_t = conv_t_count / n_t
        conv_c = conv_c_count / n_c

        count = np.array([conv_t_count, conv_c_count])
        nobs = np.array([n_t, n_c])

        z_stat, p_value = proportions_ztest(count, nobs)

        rpu_t = treatment["spend"].mean()
        rpu_c = control["spend"].mean()

        rows.append({
            "segment_variable": segment_col,
            "segment_value": segment_value,
            "comparison": f"{treatment_name} vs {control_name}",
            "treatment_users": n_t,
            "control_users": n_c,
            "treatment_buyers": int(conv_t_count),
            "control_buyers": int(conv_c_count),
            "treatment_conversion_rate": conv_t,
            "control_conversion_rate": conv_c,
            "absolute_conversion_lift": conv_t - conv_c,
            "relative_conversion_lift": (conv_t / conv_c - 1) if conv_c > 0 else np.nan,
            "conversion_p_value": p_value,
            "treatment_revenue_per_user": rpu_t,
            "control_revenue_per_user": rpu_c,
            "revenue_per_user_lift": rpu_t - rpu_c
        })

    return pd.DataFrame(rows)

### Top Segment-Level Revenue Lift Results

The table below ranks segment-treatment combinations by revenue-per-user lift.

Segment-level results should be interpreted more cautiously than overall experiment results because smaller sample sizes reduce statistical power.

In [12]:
segment_lift_tables = []

for segment_col in ["history_segment", "zip_code", "channel", "newbie"]:
    for treatment_name in ["Mens E-Mail", "Womens E-Mail"]:
        segment_lift_tables.append(
            segment_lift_with_significance(df, segment_col, treatment_name)
        )

segment_lift_results = pd.concat(segment_lift_tables, ignore_index=True)

segment_lift_results["is_significant_05"] = (
    segment_lift_results["conversion_p_value"] < 0.05
)

segment_lift_results.sort_values(
    "revenue_per_user_lift",
    ascending=False
).head(20)

,segment_variable,segment_value,comparison,treatment_users,control_users,treatment_buyers,control_buyers,treatment_conversion_rate,control_conversion_rate,absolute_conversion_lift,relative_conversion_lift,conversion_p_value,treatment_revenue_per_user,control_revenue_per_user,revenue_per_user_lift,is_significant_05
13,history_segment,"7) $1,000 +",Womens E-Mail vs No E-Mail,428,416,11,6,0.025701,0.014423,0.011278,0.781931,2.436246e-01,4.676589,2.169808,2.506781,False
3,history_segment,4) $350 - $500,Mens E-Mail vs No E-Mail,2097,2124,42,24,0.020029,0.011299,0.008729,0.772532,2.227775e-02,2.579208,1.010268,1.568940,True
5,history_segment,"6) $750 - $1,000",Mens E-Mail vs No E-Mail,644,622,16,3,0.024845,0.004823,0.020022,4.151139,3.398796e-03,1.883680,0.349534,1.534146,True
6,history_segment,"7) $1,000 +",Mens E-Mail vs No E-Mail,464,416,11,6,0.023707,0.014423,0.009284,0.643678,3.178188e-01,3.491875,2.169808,1.322067,False
11,history_segment,5) $500 - $750,Womens E-Mail vs No E-Mail,1662,1652,26,9,0.015644,0.005448,0.010196,1.871507,4.093231e-03,1.824507,0.538856,1.285651,True
20,channel,Multichannel,Mens E-Mail vs No E-Mail,2577,2606,44,18,0.017074,0.006907,0.010167,1.471953,7.618181e-04,1.824870,0.615741,1.209129,True
23,channel,Multichannel,Womens E-Mail vs No E-Mail,2579,2606,36,18,0.013959,0.006907,0.007052,1.020938,1.238970e-02,1.773276,0.615741,1.157536,True
4,history_segment,5) $500 - $750,Mens E-Mail vs No E-Mail,1597,1652,22,9,0.013776,0.005448,0.008328,1.528630,1.464083e-02,1.641866,0.538856,1.103010,True
27,newbie,1,Mens E-Mail vs No E-Mail,10686,10695,119,40,0.011136,0.003740,0.007396,1.977506,3.096637e-10,1.332468,0.372448,0.960020,True
22,channel,Web,Mens E-Mail vs No E-Mail,9490,9373,123,54,0.012961,0.005761,0.007200,1.249696,2.927565e-07,1.521835,0.671386,0.850449,True


## 8. Export Outputs

The final summary tables are exported for reporting, dashboarding, and repository documentation.

In [13]:
from pathlib import Path

Path("../outputs").mkdir(exist_ok=True)

group_summary.to_csv(
    "../outputs/experiment_group_summary.csv",
    index=False
)

comparisons.to_csv(
    "../outputs/experiment_lift_results.csv",
    index=False
)

bootstrap_revenue_lift.to_csv(
    "../outputs/experiment_revenue_bootstrap_ci.csv",
    index=False
)

segment_lift_results.to_csv(
    "../outputs/experiment_segment_lift.csv",
    index=False
)

# Experimentation Summary

## Objective

The goal of this phase was to evaluate whether an email marketing campaign increased customer visits, conversions, and revenue compared with a randomized no-email control group.

The experiment included three groups:

- Mens E-Mail
- Womens E-Mail
- No E-Mail control

The primary business question was whether sending promotional email campaigns created incremental conversion and revenue lift.

---

## Experiment Design

The dataset contains 64,000 customers randomly assigned into campaign groups.

Each customer has pre-campaign attributes such as:

- Recency
- Historical spend segment
- Historical spend amount
- Zip-code type
- New customer flag
- Channel

Post-campaign outcomes include:

- Visit
- Conversion
- Spend

The no-email group was used as the control group.

---

## Overall Results

Both email treatments outperformed the no-email control group.

| Comparison | Conversion Rate Lift | Relative Lift | Revenue/User Lift | Interpretation |
|---|---:|---:|---:|---|
| Mens E-Mail vs Control | +0.68 percentage points | +118.8% | +$0.77 | Strongest treatment |
| Womens E-Mail vs Control | +0.31 percentage points | +54.3% | +$0.42 | Positive treatment effect |
| Any E-Mail vs Control | +0.50 percentage points | +86.5% | +$0.60 | Campaign overall worked |

Mens E-Mail generated the largest lift across both conversion and revenue per user.

---

## Statistical Significance

Two-proportion z-tests showed that conversion lift was statistically significant for all major comparisons.

- Mens E-Mail vs Control: highly significant
- Womens E-Mail vs Control: significant
- Any E-Mail vs Control: highly significant

This suggests that the observed conversion lift is unlikely to be random noise.

---

## Revenue Lift

Bootstrap confidence intervals were used to evaluate revenue per user lift because spend is highly skewed.

The 95% confidence intervals for revenue lift were fully above zero:

| Comparison | Revenue/User Lift | 95% Confidence Interval |
|---|---:|---:|
| Mens E-Mail vs Control | +$0.77 | $0.48 to $1.05 |
| Womens E-Mail vs Control | +$0.42 | $0.16 to $0.68 |
| Any E-Mail vs Control | +$0.60 | $0.38 to $0.82 |

This confirms that the campaign increased revenue per user, not just conversion rate.

---

## Segment-Level Treatment Effects

Treatment effects varied meaningfully across customer segments.

Mens E-Mail showed the broadest statistically reliable lift across:

- Mid-to-high historical spend customers
- Multichannel customers
- New customers
- Web, phone, and suburban customer segments

Womens E-Mail showed strong lift in some higher historical spend groups, but several of the largest revenue-lift estimates were directional rather than statistically significant due to smaller sample sizes.

Strong statistically significant segment lifts included:

- Mens E-Mail among customers with $350–$500 history
- Mens E-Mail among customers with $750–$1,000 history
- Womens E-Mail among customers with $500–$750 history
- Mens and Womens E-Mail among multichannel customers
- Mens E-Mail among new customers

---

## Business Recommendations

The campaign should be considered successful.

Recommended actions:

- Prioritize Mens E-Mail as the primary campaign variant because it produced the strongest overall conversion and revenue lift.
- Continue using Womens E-Mail selectively, especially for customer groups where it showed meaningful lift.
- Target multichannel customers more aggressively because both email variants performed well in this segment.
- Test more personalized messaging for high-history customers, where revenue lift was directionally strong but sometimes not statistically conclusive.
- Use historical spend and channel as targeting dimensions for future campaign experimentation.

---

## Limitations

Some segment-level results should be interpreted cautiously.

Reasons:

- Smaller segment sizes reduce statistical power.
- Revenue per user is skewed because most users spend zero.
- Some large revenue-lift estimates were not statistically significant.
- Segment results should guide future tests rather than be treated as final causal conclusions.

---

## Conclusion

The email experiment produced statistically significant conversion and revenue lift.

Mens E-Mail was the strongest overall treatment, while Womens E-Mail also generated positive lift.

The analysis demonstrates how experimentation can support business decisions through treatment-control comparison, lift estimation, confidence intervals, and heterogeneous treatment-effect analysis.